In [79]:
import pandas as pd
import numpy as np

## **Zinn+ 2022**

### Data Description

Table 1:

```

```

### CDS X-Matching

Read-in xmatch.

In [80]:
array_full = pd.read_csv("../0_inputdata_asteroseismic/Zinn2022_cdsxmatch.csv",
                         dtype={"Source": "Int64"}, low_memory=False)

Convert EV state, with:

 - 0 ==> not specified
 - 1 ==> likely RGB
 - 2 ==> likely CHeB

In [81]:
array_full["EV"] = array_full["EV"].replace({
    "nan": np.nan,
    "RGB": int(1),
    "RGB/AGB": int(0),
    "RC": int(2)
})

In [82]:
astero_avg = pd.read_csv(
    "../0_inputdata_asteroseismic/zinn22_avg_dnunumax.tsv",
    sep="\t",
    comment="#",
    usecols=lambda c: c not in ["_RAJ2000", "_DEJ2000"],
    na_values=["", " "],
)

# drop header lines/rows
astero_avg = astero_avg.iloc[2:].reset_index(drop=True)

# force columns to int/float
for col in astero_avg.columns:
  if col == "EPIC":
    astero_avg[col] = pd.to_numeric(astero_avg[col], errors="coerce").astype(
        "Int64"
    )
  else:
    astero_avg[col] = pd.to_numeric(astero_avg[col], errors="coerce")

astero_avg["e_dnu"] = astero_avg["e_dnu-c"] / astero_avg["XSharma"] # raw dnu uncertainty from XSharma
astero_avg = astero_avg.drop(columns=["e_dnu-c", "XSharma"]) # drop unnecessary data
astero_avg.columns = astero_avg.columns.str.replace(r"-(c|u)$", "", regex=True) # remove -c or -u
astero_avg = astero_avg[["EPIC", "numax", "e_numax", "dnu", "e_dnu"]]
astero_avg = astero_avg.dropna()

print(astero_avg.head())


        EPIC   numax  e_numax    dnu     e_dnu
0  201051625  18.081    0.096  2.579  0.187078
1  201079541  40.223    0.492  4.568  0.060194
2  201082897  32.143    0.273  4.353  0.094246
3  201084449  29.515    0.439  3.597  0.055179
4  201084664  16.317    0.434  2.590  0.146971


In [83]:
# def mean_std_func(values, errors):
#     vals = np.stack([array_full[c] for c in values])
#     errs = np.stack([array_full[c] for c in errors])

#     errs[(errs <= 0) | ~np.isfinite(errs)] = np.nan # set invalid uncertainties to NaN

#     vals = np.where(np.isfinite(errs), vals, np.nan)

#     mean = np.nanmean(vals, axis=0)
#     std = np.nanstd(errs, axis=0)

#     return mean, std

In [84]:
# numax_cols = ["numax-A2Z-o", "numax-BAM-o", "numax-BHM-o", "numax-CAN-o", "numax-COR-o", "numax-SYD-o"]
# enumax_cols = ["e_numax-A2Z-o", "e_numax-BAM-o", "e_numax-BHM-o", "e_numax-CAN-o", "e_numax-COR-o", "e_numax-SYD-o"]

# dnu_cols = ["dnu-A2Z-o", "dnu-BAM-o", "dnu-BHM-o", "dnu-CAN-o", "dnu-COR-o", "dnu-SYD-o"]
# ednu_cols = ["e_dnu-A2Z-o", "e_dnu-BAM-o", "e_dnu-BHM-o", "e_dnu-CAN-o", "e_dnu-COR-o", "e_dnu-SYD-o"]

# numax, e_numax = mean_std_func(numax_cols, enumax_cols)
# dnu, e_dnu = mean_std_func(dnu_cols, ednu_cols)

In [85]:
array_full = pd.merge(array_full, astero_avg, on="EPIC", how="right")

Curate simplfied array.

In [86]:
z22 = pd.DataFrame({
    # star identifiers
    "gaiadr3_id": array_full["Source"],
    "ra": array_full["RAdeg"],
    "dec": array_full["DEdeg"],
    # asteroseismic parameters
    "numax": array_full["numax"],
    "e_numax": array_full["e_numax"],
    "dnu": array_full["dnu"],
    "e_dnu": array_full["e_dnu"],
    "ev_state": array_full["EV"],
    # Gaia parameters
    "GAIA-teff": array_full["Teff"],
    "GAIA-e_teff": (array_full["B_Teff"] - array_full["b_Teff"]) / 2,
    "GAIA-logg": array_full["logg"],
    "GAIA-e_logg": (array_full["B_logg"] - array_full["b_logg"]) / 2,
    "GAIA-fe_h": array_full["[Fe/H]"],
    "GAIA-e_fe_h": (array_full["B_[Fe/H]"] - array_full["b_[Fe/H]"]) / 2,
    # datasets
    "h23": np.full(len(array_full), int(0)),
    "s26": np.full(len(array_full), int(0)),
    "w26": np.full(len(array_full), int(0)),
    "y18": np.full(len(array_full), int(0)),
    "z22": np.full(len(array_full), int(1)),
    # GSP Spec parameters
    "GSPSPEC-teff": array_full["Teff-S"],
    "GSPSPEC-e_teff": (array_full["B_Teff-S"] - array_full["b_Teff-S"]) / 2,
    "GSPSPEC-logg": array_full["logg-S"],
    "GSPSPEC-e_logg": (array_full["B_logg-S"] - array_full["b_logg-S"]) / 2,
    "GSPSPEC-fe_h": array_full["[Fe/H]-S"],
    "GSPSPEC-e_fe_h": (array_full["B_[Fe/H]-S"] - array_full["b_[Fe/H]-S"]) / 2,
    "GSPSPEC-si_fe": array_full["[Si/Fe]-S"],
    "GSPSPEC-e_si_fe": (array_full["B_[Si/Fe]-S"] - array_full["b_[Si/Fe]-S"]) / 2,
    "GSPSPEC-ca_fe": array_full["[Ca/Fe]-S"],
    "GSPSPEC-e_ca_fe": (array_full["B_[Ca/Fe]-S"] - array_full["b_[Ca/Fe]-S"]) / 2,
    "GSPSPEC-ti_fe": array_full["[Ti/Fe]-S"],
    "GSPSPEC-e_ti_fe": (array_full["B_[Ti/Fe]-S"] - array_full["b_[Ti/Fe]-S"]) / 2,
    "GSPSPEC-mg_fe": array_full["[Mg/Fe]-S"],
    "GSPSPEC-e_mg_fe": (array_full["B_[Mg/Fe]-S"] - array_full["b_[Mg/Fe]-S"]) / 2,
    "GSPSPEC-nd_fe": array_full["[Nd/Fe]-S"],
    "GSPSPEC-e_nd_fe": (array_full["B_[Nd/Fe]-S"] - array_full["b_[Nd/Fe]-S"]) / 2,
    "GSPSPEC-s_fe": array_full["[S/Fe]-S"],
    "GSPSPEC-e_s_fe": (array_full["B_[S/Fe]-S"] - array_full["b_[S/Fe]-S"]) / 2,
    "GSPSPEC-zr_fe": array_full["[Zr/Fe]-S"],
    "GSPSPEC-e_zr_fe": (array_full["B_[Zr/Fe]-S"] - array_full["b_[Zr/Fe]-S"]) / 2,
    "GSPSPEC-n_fe": array_full["[N/Fe]-S"],
    "GSPSPEC-e_n_fe": (array_full["B_[N/Fe]-S"] - array_full["b_[N/Fe]-S"]) / 2,
    "GSPSPEC-cr_fe": array_full["[Cr/Fe]-S"],
    "GSPSPEC-e_cr_fe": (array_full["B_[Cr/Fe]-S"] - array_full["b_[Cr/Fe]-S"]) / 2,
    "GSPSPEC-ce_fe": array_full["[Ce/Fe]-S"],
    "GSPSPEC-e_ce_fe": (array_full["B_[Ce/Fe]-S"] - array_full["b_[Ce/Fe]-S"]) / 2,
    "GSPSPEC-ni_fe": array_full["[Ni/Fe]-S"],
    "GSPSPEC-e_ni_fe": (array_full["B_[Ni/Fe]-S"] - array_full["b_[Ni/Fe]-S"]) / 2
}).astype({"gaiadr3_id": "Int64"})

z22 = z22.dropna(subset=["e_numax", "e_dnu"])

Taken within $0.75\times$ and $1.35\times$ of $\Delta\nu$-$\nu_\mathrm{max}$ relation (Stello+ 09, Mosser+ 11):

$$ \Delta\nu = 0.27 \nu_\mathrm{max}^{0.76}. $$

In [87]:
def relation(numax):
    coeff = 0.27
    return coeff * numax**(0.76)

mask = (z22["dnu"] < 1.35*relation(z22["numax"])) & (z22["dnu"] > 0.75*relation(z22["numax"]))

z22 = z22[mask]

Write to `.csv` file.

In [88]:
z22.to_csv("./zinn22_asteroseismic.csv", index=False)

### **Identify Any Differing RA and Dec**

In [89]:
valid = z22.dropna(subset=["gaiadr3_id"])

ra_diff = valid.groupby("gaiadr3_id")["ra"].transform("nunique") > 1
dec_diff = valid.groupby("gaiadr3_id")["dec"].transform("nunique") > 1
coord_dups = valid[ra_diff | dec_diff].sort_values("gaiadr3_id")

print(f"IDs with differing (ra, dec): {coord_dups['gaiadr3_id'].nunique()} ({len(coord_dups)} rows)")

for gid, group in coord_dups.groupby("gaiadr3_id"):
    varying_cols = [c for c in group.columns if group[c].nunique(dropna=False) > 1]
    print(f"\nID: {int(gid)}")
    print(group[["ra", "dec"] + [c for c in varying_cols if c not in ("ra", "dec")]].to_string(index=False, float_format="%.10f"))


IDs with differing (ra, dec): 0 (0 rows)


In [90]:
# ra_diff = z22.groupby("gaiadr3_id")["ra"].transform("nunique") > 1
# dec_diff = z22.groupby("gaiadr3_id")["dec"].transform("nunique") > 1

# z22 = z22[~(ra_diff | dec_diff)].copy()

# # z22.to_csv("./zinn22_asteroseismic.csv", index=False)